# Energy-portfolio-risk-analysis-tool-using-Python-for-MtM-VaR-stress-testing-and-GM-R-calculations.

## ライブラリの読み込み

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats

## 分析の設定を定義

In [2]:
np.random.seed(42)

# 分析期間を定義する
start_date = "2023-01-01"
end_date = "2025-12-31"

dates = pd.date_range(start=start_date, end=end_date, freq="D")
n = len(dates)

## コモディティ価格の定義

In [3]:
# シミュレーション価格を生成する関数を定義
def simulate_prices(
    initial_price: int, volatility: float, trend: float, n: int
) -> list:
    return np.exp(np.cumsum(np.random.normal(trend, volatility, n))) * initial_price

In [4]:
# 各種価格を定義
oil_prices = simulate_prices(70, 0.02, 0.0001, n)
gas_prices = simulate_prices(3, 0.03, 0.0002, n)
renewable_prices = simulate_prices(50, 0.015, 0.0003, n)

In [6]:
# ポートフォリオポジションを定義
positions = {"oil": 1000, "gas": 5000, "renewable": -2000}

# MtM(Market to Market)を計算する
mtm_oil = oil_prices * positions["oil"]
mtm_gas = gas_prices * positions["gas"]
mtm_renewable = renewable_prices * positions["renewable"]
total_mtm = mtm_oil + mtm_gas + mtm_renewable

## テーブルを定義

In [7]:
df = pd.DataFrame(
    {
        "Date": dates,
        "Oil Price": oil_prices,
        "Gas Price": gas_prices,
        "Renewable Price": renewable_prices,
        "MtM Oil": mtm_oil,
        "MtM Gas": mtm_gas,
        "MtM Renewable": mtm_renewable,
        "Total MtM": total_mtm,
    }
)
df.set_index("Date", inplace=True)

df.head()

,Oil Price,Gas Price,Renewable Price,MtM Oil,MtM Gas,MtM Renewable,Total MtM
Date,,,,,,,
2023-01-01,70.705936,3.007687,51.556982,70705.935670,15038.434993,-103113.964739,-17369.594075
2023-01-02,70.517735,2.833252,50.528627,70517.735044,14166.262200,-101057.253267,-16373.256024
2023-01-03,71.444292,2.912801,50.687819,71444.291764,14564.004896,-101375.638700,-15367.342039
2023-01-04,73.661377,2.943825,50.202052,73661.377157,14719.125768,-100404.103719,-12023.600793
2023-01-05,73.324555,3.033904,50.538944,73324.554524,15169.518068,-101077.888025,-12583.815433


## VaR(Value at Risk)を計算する

In [ ]:
def calculate_var(returns: list, confidence_level: int) -> float:
    return np.percentile(returns, 100 - confidence_level)

In [9]:
returns = df["Total MtM"].pct_change().dropna()

var_95 = calculate_var(returns, 95)
var_90 = calculate_var(returns, 90)

print(var_95)

-0.28366199880701193


## Stress Testingを計算する

In [10]:
def stress_test(df, scenario):
    stressed_df = df.copy()
    for commodity, change in scenario.items():
        col_name = f"{commodity.capitalize()} Price"
        stressed_df[col_name] *= 1 + change
        stressed_df[f"MtM {commodity.capitalize()}"] = (
            stressed_df[col_name] * positions[commodity.lower()]
        )
    stressed_df["Total MtM"] = stressed_df[
        [f"MtM {c.capitalize()}" for c in positions.keys()]
    ].sum(axis=1)
    return stressed_df

In [11]:
# Define stress scenarios
scenarios = {
    "base": {"oil": 0, "gas": 0, "renewable": 0},
    "oil_spike": {"oil": 0.5, "gas": 0.1, "renewable": 0.05},
    "gas_crash": {"oil": -0.1, "gas": -0.3, "renewable": 0.1},
    "renewable_boom": {"oil": -0.2, "gas": -0.15, "renewable": 0.4},
}

In [12]:
# Run stress tests
stress_results = {
    name: stress_test(df, scenario) for name, scenario in scenarios.items()
}

print(stress_results)

{'base':              Oil Price  Gas Price  Renewable Price        MtM Oil  \
Date                                                                
2023-01-01   70.705936   3.007687        51.556982   70705.935670   
2023-01-02   70.517735   2.833252        50.528627   70517.735044   
2023-01-03   71.444292   2.912801        50.687819   71444.291764   
2023-01-04   73.661377   2.943825        50.202052   73661.377157   
2023-01-05   73.324555   3.033904        50.538944   73324.554524   
...                ...        ...              ...            ...   
2025-12-27  155.896167  12.956546        75.455885  155896.167397   
2025-12-28  158.197268  12.793329        73.731181  158197.267525   
2025-12-29  158.377544  13.023826        75.435432  158377.543836   
2025-12-30  160.731376  12.416630        75.085336  160731.376431   
2025-12-31  160.488160  12.592506        74.869776  160488.160030   

                 MtM Gas  MtM Renewable     Total MtM  
Date                                 